In [1]:
import torch
import torch.nn as nn
from torch.nn import functional as F
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)
block_size = 8
batch_size = 4
max_iters = 1000
# eval_interval = 2500
learning_rate = 4e-2
eval_iters = 250

cuda


In [2]:
with open('inputs/wizard_of_oz.txt', 'r', encoding='utf-8') as f:
    text = f.read()

print(len(text))
print(text[:200])

226617
﻿CHAPTER 1.

THE EARTHQUAKE


The train from 'Frisco was very late. It should have arrived at Hugson's
siding at midnight, but it was already five o'clock and the gray dawn
was breaking in the east wh


In [105]:
# Added data preprocessing to mremove unwanted characters
remove_chars = ":;?!&(),-.[]_'\n\"\ufeff"

# Create a translation table to remove specific characters
translator = str.maketrans('', '', remove_chars)

# Apply translation
text = text.translate(translator)

In [106]:
chars = sorted(set(text))
print(chars)
print(len(chars))
vocab_size = len(chars)

[' ', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']
63


Tokenizer contains an encoder and a decoder

what encoder does it covnerts each element of array to integer

In [107]:
# basic char level tokenizer
# takes a char and covnerts into a integer

string_to_int = {ch:i for i,ch in enumerate(chars)}
int_to_string = {i:ch for i,ch in enumerate(chars)}

encode = lambda s: [string_to_int[c] for c in s]
decode = lambda l: ''.join(int_to_string[i] for i in l)

encoded_hello = encode('hello')
print(encoded_hello)
decoded_hello = decode(encoded_hello)
print(decoded_hello)

[44, 41, 48, 48, 51]
hello


Tokenizer contains an encoder and a decoder

what encoder does it covnerts each element of array to integer

In [108]:
# basic char level tokenizer
# takes a char and covnerts into a integer

string_to_int = {ch:i for i,ch in enumerate(chars)}
int_to_string = {i:ch for i,ch in enumerate(chars)}

encode = lambda s: [string_to_int[c] for c in s]
decode = lambda l: ''.join(int_to_string[i] for i in l)

encoded_hello = encode('hello')
print(encoded_hello)
decoded_hello = decode(encoded_hello)
print(decoded_hello)


[44, 41, 48, 48, 51]
hello


In [109]:
# converting to tensor type
data = torch.tensor(encode(text), dtype=torch.long)
print(data[:100])

tensor([13, 18, 11, 26, 30, 15, 28,  0,  2, 30, 18, 15,  0, 15, 11, 28, 30, 18,
        27, 31, 11, 21, 15, 30, 44, 41,  0, 56, 54, 37, 45, 50,  0, 42, 54, 51,
        49,  0, 16, 54, 45, 55, 39, 51,  0, 59, 37, 55,  0, 58, 41, 54, 61,  0,
        48, 37, 56, 41,  0, 19, 56,  0, 55, 44, 51, 57, 48, 40,  0, 44, 37, 58,
        41,  0, 37, 54, 54, 45, 58, 41, 40,  0, 37, 56,  0, 18, 57, 43, 55, 51,
        50, 55, 55, 45, 40, 45, 50, 43,  0, 37])


In [110]:
# # train and val data split
# n = int(0.8*len(data))
# train_data = data[:n]
# val_data = data[n:]

In [111]:
# train_data

In [112]:
# x = train_data[:block_size]
# y = train_data [1:block_size+1]

# for t in range(block_size):
#     context = x[:t+1]
#     target = y[t]
#     print('when  input is ', context, 'target is ', target)

In [113]:
# train and val data split
n = int(0.8*len(data))
train_data = data[:n]
val_data = data[n:]

def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    # print(ix)
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

x, y = get_batch('train')
print('inputs:')
# print(x.shape)
print(x)
print('targets:')
print(y)

inputs:
tensor([[59,  0, 37, 54, 54, 45, 58, 37],
        [41, 37, 56,  0, 52, 41, 51, 52],
        [56, 44, 41,  0, 54, 51, 51, 42],
        [43, 50,  0, 54, 41, 37, 40, 45]], device='cuda:0')
targets:
tensor([[ 0, 37, 54, 54, 45, 58, 37, 48],
        [37, 56,  0, 52, 41, 51, 52, 48],
        [44, 41,  0, 54, 51, 51, 42,  0],
        [50,  0, 54, 41, 37, 40, 45, 50]], device='cuda:0')


In [114]:
# this decorator makes sure pytorch does not uses gratients at all in here, that will reduce computation & memory usage
# so it's overall better for performance, and becoz we are just reporting a loss, we dont really need to do any optimizing or gradient conputation here
# for any outside funtion which is used for evalutation and where model is being passed and we dont want to use gradients then use this decorator
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

In [115]:
class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)
        
    def forward(self, index, targets=None):
        logits = self.token_embedding_table(index)
        
        
        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        
        return logits, loss
    
    def generate(self, index, max_new_tokens):
        # index is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # get the predictions
            logits, loss = self.forward(index)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            index_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            index = torch.cat((index, index_next), dim=1) # (B, T+1)
        return index

model = BigramLanguageModel(vocab_size)
m = model.to(device)

context = torch.zeros((1,1), dtype=torch.long, device=device)
generated_chars = decode(m.generate(context, max_new_tokens=500)[0].tolist())
print(generated_chars)

 21FOzVFW81PRPZfakoGxqjvg7xwjaohBqtfirHjxO32V9mC6PLs3upmSC54cvYsExLjxfKuOwdGQR7eBBBA1pjZPTiCSB1FiTUt81KIq0x6dfolw5cmqMXOyfFi9xuNKIJxZPGcIhIJxz70PjrKwEtXpIsiYKOyY0H5EZ1FLJSrnFAw4nrCimSYk8rRzoq3cLV7V9qo7xOqoh Uel 275wSjtXNelPT3vSqnFnUteOzqnxuSOwP024prbZm 0priRzonC3lcdm5LrxJhtjBG8cyfzcprbu8LuF1fLsg3MDaZCjWKhaEMXTO5JecQU Lsxb12jvHx3Bjc5tb32Da50yNRbNRBcdlX 7wUxNtBrvXuStoLsNni  04yh2O9SqOTD1UKldwJxTlgxNtsB0pU EM5ktZur fUr1FW5Nq6nUwprSPd22oiqxu UmbIgWqoLuSp5kwwxadpfxbM8VNtevU922eqfkOxBJGQh5DLBdzMLsBMXhx


In [116]:
# create a PyTorch optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):
    if iter % eval_iters == 0:
        losses = estimate_loss()
        print(f"step: {iter}, train loss: {losses['train']:.3f}, val loss: {losses['val']:.3f}")

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = model.forward(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
print(loss.item())

step: 0, train loss: 4.484, val loss: 4.481
step: 250, train loss: 2.502, val loss: 2.558
step: 500, train loss: 2.439, val loss: 2.481
step: 750, train loss: 2.411, val loss: 2.477
2.453925132751465


In [ ]:
context = torch.zeros((1,1), dtype=torch.long, device=device)
generated_chars = decode(m.generate(context, max_new_tokens=500)[0].tolist())
print(generated_chars)

need to familiarize audience with optimizers (AdamW, Adam, SGD, MSE…) no need to jump into the formulas, just what the optimizer does for us and some of the differences/similarities between them

Mean Squared Error (MSE): MSE is a common loss function used in regression problems, where the goal is to predict a continuous output. It measures the average squared difference between the predicted and actual values, and is often used to train neural networks for regression tasks.
Gradient Descent (GD): is an optimization algorithm used to minimize the loss function of a machine learning model. The loss function measures how well the model is able to predict the target variable based on the input features. The idea of GD is to iteratively adjust the model parameters in the direction of the steepest descent of the loss function
Momentum: Momentum is an extension of SGD that adds a "momentum" term to the parameter updates. This term helps smooth out the updates and allows the optimizer to continue moving in the right direction, even if the gradient changes direction or varies in magnitude. Momentum is particularly useful for training deep neural networks.
RMSprop: RMSprop is an optimization algorithm that uses a moving average of the squared gradient to adapt the learning rate of each parameter. This helps to avoid oscillations in the parameter updates and can improve convergence in some cases.
Adam: Adam is a popular optimization algorithm that combines the ideas of momentum and RMSprop. It uses a moving average of both the gradient and its squared value to adapt the learning rate of each parameter. Adam is often used as a default optimizer for deep learning models.
AdamW: AdamW is a modification of the Adam optimizer that adds weight decay to the parameter updates. This helps to regularize the model and can improve generalization performance. We will be using the AdamW optimizer as it best suits the properties of the model we will train in this video.
find more optimizers and details at torch.optim